# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, their `@id`s, and associated fields. Each entity is referenced by its `@id` for reproducibility and traceability.

In [ ]:
# Explore all record sets and their fields in the dataset
print("Available Record Sets and their fields:")
record_sets = [rs for rs in metadata.record_set]
for rs in record_sets:
    print(f"- Record Set Name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.field:
        print(f"    - Field: {field.name}, @id: {field.id}")
    print("")

Let's print the first record from each record set using its `@id`. This helps to understand the data structure.

In [ ]:
# Show a sample record from each record set, referencing by @id
for rs in record_sets:
    print(f"Sample record from record set: {rs.name} (@id: {rs.id})")
    it = dataset.records(record_set=rs.id)
    try:
        sample = next(it)
        print(sample)
    except StopIteration:
        print("No records found.")
    print("")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for further analysis. Entities are referenced using their `@id`. Modify as needed for your analysis.

In [ ]:
# Extract data from each record set into DataFrames
dataframes = {}
for rs in record_sets:
    # Use @id as key for mapping
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} rows for Record Set '@id': {rs.id} (name: {rs.name})")

# Show columns and a preview for the first available record set
if record_sets:
    first_rs_id = record_sets[0].id
    print(f"\nColumns in record set '@id': {first_rs_id}")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data, and grouping by key fields.

By referencing the field `@id`s, your analysis remains robust and reproducible. We'll perform basic operations on the first record set. Update which record set and field as your analysis requires.

In [ ]:
from pandas.api.types import is_numeric_dtype

# Use the first record set for EDA
record_set_id = first_rs_id
df = dataframes[record_set_id]

# Identify numeric fields/columns by dtype
numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col])]
print(f"Numeric fields in record set '@id': {record_set_id}\n{numeric_fields}\n")

if numeric_fields:
    numeric_field = numeric_fields[0]  # Use first available numeric field for demonstration. Override as needed.
    threshold = df[numeric_field].mean()  # Use mean as threshold for demonstration
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a non-numeric field if available
    group_fields = [col for col in df.columns if not is_numeric_dtype(df[col])]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped means of {numeric_field} by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships using matplotlib or seaborn. Numeric columns from the first record set are plotted as examples.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' in record set '@id': {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot for the first two numeric fields (if available)
    if len(numeric_fields) >= 2:
        plt.figure(figsize=(7, 5))
        sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"Scatter plot of {numeric_fields[0]} vs {numeric_fields[1]}")
        plt.show()
else:
    print("No numeric fields found for visualization.")

## 6. Conclusion
This notebook demonstrated how to:
- Load and review metadata from a Croissant dataset
- Explore record sets and field structure by their `@id`
- Extract data for analysis in pandas DataFrames
- Filter, normalize, group, and visualize records for exploratory analysis

Repeat and adapt this workflow for other record sets and data fields as required by your research.